<a href="https://colab.research.google.com/github/rafiebazine/rag-image-captioning/blob/rafie/phase_01_data_exploration/data_exploration_of_sub_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Exploration
Calculation of metrics
- Make sure to add the split files to the splits folder `splits/<sub-sets>.json`

## Download raw data-sets if needed

In [1]:
# 1. Download annotations
!wget -q --show-progress http://images.cocodataset.org/annotations/annotations_trainval2017.zip


# 2. Extract all downloaded zip files
print("Extracting annotations...")
!unzip -q annotations_trainval2017.zip


annotations_trainva 100%[===================>] 241.19M  20.9MB/s    in 13s     
train2017.zip         7%[>                   ]   1.34G  13.4MB/s    eta 37m 19s^C
val2017.zip         100%[===================>] 777.80M  16.6MB/s    in 56s     
Extracting annotations...
Extracting training dataset (this may take a while)...
[train2017.zip]
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of train2017.zip or
        train2017.zip.zip, and cannot find train2017.zip.ZIP, period.
Extracting validation dataset...


## Distribution of object classes

In [ ]:
from collections import Counter
import json
import os

# Google Colab path definitions
INSTANCES_TRAIN_FILE = os.path.join("annotations", "instances_train2017.json")
INSTANCES_VAL_FILE = os.path.join("annotations", "instances_val2017.json")

TEST_SPLIT_FILE = os.path.join("splits", "test_100.json")
TEST_1000_SPLIT_FILE = os.path.join("splits", "test_1000.json")
VAL_SPLIT_FILE = os.path.join("splits", "val_100.json")
KB_SPLIT_FILE = os.path.join("splits", "kb_10k.json")


def read_json(file_path):
    """Utility function to load a JSON file."""
    with open(file_path, "r") as f:
        return json.load(f)


def build_image_to_categories_map(instances_data):
    """Maps image_id -> list of object category IDs present in that image."""
    image_to_categories = {}
    for ann in instances_data["annotations"]:
        image_id = ann["image_id"]
        cat_id = ann["category_id"]
        image_to_categories.setdefault(image_id, []).append(cat_id)
    return image_to_categories


def count_top_classes_for_split(
    split_data, image_to_categories_map, category_name_map, top_k=50
):
    """Calculates object instance frequencies for a split and returns Top K."""
    class_counts = Counter()
    for item in split_data:
        image_id = item["image_id"]
        # Extract the categories existing in the current image
        cat_ids = image_to_categories_map.get(image_id, [])
        for cid in cat_ids:
            # Get category name
            cat_name = category_name_map.get(cid, f"Unknown_{cid}")
            # Increase category count
            class_counts[cat_name] += 1
    # order (descend) and return top k
    return class_counts.most_common(top_k)


def display_top_classes(split_name, top_classes):
    """Prints a formatted table of Top 50 classes and their occurrence counts."""
    print(f"\n==================================================")
    print(f"   Top 50 Object Classes: {split_name}")
    print(f"==================================================")
    print(f"{'Rank':<6} | {'Category Name':<22} | {'Occurrences':<12}")
    print("-" * 48)
    for rank, (cat_name, count) in enumerate(top_classes, start=1):
        print(f"{rank:<6} | {cat_name:<22} | {count:<12}")


def run_object_distribution_analysis():
    print("Loading COCO instance annotations...")
    train_instances = read_json(INSTANCES_TRAIN_FILE)
    val_instances = read_json(INSTANCES_VAL_FILE)

    # Build category ID to Name map
    category_name_map = {
        cat["id"]: cat["name"] for cat in train_instances["categories"]
    }

    # Build image_id -> category_ids lookup maps
    train_image_to_cats = build_image_to_categories_map(train_instances)
    val_image_to_cats = build_image_to_categories_map(val_instances)

    # 1. Process Test Split (sourced from val2017)
    if os.path.exists(TEST_SPLIT_FILE):
        test_data = read_json(TEST_SPLIT_FILE)
        top_test = count_top_classes_for_split(
            test_data, val_image_to_cats, category_name_map
        )
        display_top_classes("Test Subset (test_100.json)", top_test)
    else:
        print(f"⚠️ File not found: {TEST_SPLIT_FILE}")

    # 1b. Process Test 1000 Split (sourced from val2017)
    if os.path.exists(TEST_1000_SPLIT_FILE):
        test_1000_data = read_json(TEST_1000_SPLIT_FILE)
        top_test_1000 = count_top_classes_for_split(
            test_1000_data, val_image_to_cats, category_name_map
        )
        display_top_classes("Test Subset (test_1000.json)", top_test_1000)
    else:
        print(f"⚠️ File not found: {TEST_1000_SPLIT_FILE}")

    # 2. Process Validation Split (sourced from train2017)
    if os.path.exists(VAL_SPLIT_FILE):
        val_data = read_json(VAL_SPLIT_FILE)
        top_val = count_top_classes_for_split(
            val_data, train_image_to_cats, category_name_map
        )
        display_top_classes("Validation Subset (val_100.json)", top_val)
    else:
        print(f"⚠️ File not found: {VAL_SPLIT_FILE}")

    # 3. Process Knowledge Base Split (sourced from train2017)
    if os.path.exists(KB_SPLIT_FILE):
        kb_data = read_json(KB_SPLIT_FILE)
        top_kb = count_top_classes_for_split(
            kb_data, train_image_to_cats, category_name_map
        )
        display_top_classes("Knowledge Base Subset (kb_10k.json)", top_kb)
    else:
        print(f"⚠️ File not found: {KB_SPLIT_FILE}")


if __name__ == "__main__":
    run_object_distribution_analysis()

Loading COCO instance annotations...

   Top 50 Object Classes: Test Subset (test_100.json)
Rank   | Category Name          | Occurrences 
------------------------------------------------
1      | person                 | 92          
2      | vase                   | 32          
3      | book                   | 29          
4      | bottle                 | 26          
5      | chair                  | 22          
6      | toilet                 | 18          
7      | bird                   | 17          
8      | banana                 | 17          
9      | clock                  | 16          
10     | traffic light          | 16          
11     | baseball bat           | 15          
12     | boat                   | 14          
13     | pizza                  | 13          
14     | sheep                  | 13          
15     | car                    | 13          
16     | zebra                  | 12          
17     | cup                    | 12          
18     | gira

## Scene complexity

In [ ]:
import json
import os
import statistics

# Paths for Google Colab environment
INSTANCES_TRAIN_FILE = os.path.join("annotations", "instances_train2017.json")
INSTANCES_VAL_FILE = os.path.join("annotations", "instances_val2017.json")

TEST_SPLIT_FILE = os.path.join("splits", "test_100.json")
TEST_1000_SPLIT_FILE = os.path.join("splits", "test_1000.json")
VAL_SPLIT_FILE = os.path.join("splits", "val_100.json")
KB_SPLIT_FILE = os.path.join("splits", "kb_10k.json")


def read_json(file_path):
    """Utility function to load a JSON file."""
    with open(file_path, "r") as f:
        return json.load(f)


def build_image_object_counts(instances_data):
    """Maps image_id -> total count of annotated object instances."""
    object_counts = {}
    for ann in instances_data["annotations"]:
        image_id = ann["image_id"]
        object_counts[image_id] = object_counts.get(image_id, 0) + 1
    return object_counts


def analyze_split_complexity(split_name, split_data, image_object_counts):
    """Calculates statistical distribution and simple vs. complex breakdown."""
    counts = []
    for item in split_data:
        image_id = item["image_id"]
        # Retrieve total instances for this image (default to 0 if none found)
        count = image_object_counts.get(image_id, 0)
        counts.append(count)

    total_images = len(counts)
    if total_images == 0:
        print(f"⚠️ No images found in {split_name}")
        return

    # Categorical breakdown
    simple_count = sum(1 for c in counts if 1 <= c <= 2)
    complex_count = sum(1 for c in counts if c >= 3)
    other_count = sum(1 for c in counts if c == 0)

    simple_pct = (simple_count / total_images) * 100
    complex_pct = (complex_count / total_images) * 100

    # Statistical distribution
    mean_val = statistics.mean(counts)
    median_val = statistics.median(counts)
    min_val = min(counts)
    max_val = max(counts)

    # Display results
    print(f"\n==================================================")
    print(f"   Scene Complexity Analysis: {split_name}")
    print(f"==================================================")
    print(f"Total Images Analyzed: {total_images}")
    print("-" * 50)
    print("📊 Categorical Breakdown:")
    print(f"  • Simple Scenes (1-2 objects): {simple_count:<5} ({simple_pct:.2f}%)")
    print(f"  • Complex Scenes (>=3 objects): {complex_count:<5} ({complex_pct:.2f}%)")
    if other_count > 0:
        print(f"  • Unannotated Scenes (0 objects): {other_count:<5}")
    print("-" * 50)
    print("📈 Statistical Metrics:")
    print(f"  • Mean (Average) Objects / Image: {mean_val:.2f}")
    print(f"  • Median Objects / Image:        {median_val:.2f}")
    print(f"  • Min Objects in an Image:       {min_val}")
    print(f"  • Max Objects in an Image:       {max_val}")


def run_scene_complexity_analysis():
    print("Loading COCO instance annotations...")
    train_instances = read_json(INSTANCES_TRAIN_FILE)
    val_instances = read_json(INSTANCES_VAL_FILE)

    # Build image_id -> object count maps
    train_counts_map = build_image_object_counts(train_instances)
    val_counts_map = build_image_object_counts(val_instances)

    # 1. Process Test Split (sourced from val2017)
    if os.path.exists(TEST_SPLIT_FILE):
        test_data = read_json(TEST_SPLIT_FILE)
        analyze_split_complexity("Test Subset (test_100.json)", test_data, val_counts_map)
    else:
        print(f"⚠️ File not found: {TEST_SPLIT_FILE}")

    # 1b. Process Test 1000 Split (sourced from val2017)
    if os.path.exists(TEST_1000_SPLIT_FILE):
        test_1000_data = read_json(TEST_1000_SPLIT_FILE)
        analyze_split_complexity("Test Subset (test_1000.json)", test_1000_data, val_counts_map)
    else:
        print(f"⚠️ File not found: {TEST_1000_SPLIT_FILE}")

    # 2. Process Validation Split (sourced from train2017)
    if os.path.exists(VAL_SPLIT_FILE):
        val_data = read_json(VAL_SPLIT_FILE)
        analyze_split_complexity("Validation Subset (val_100.json)", val_data, train_counts_map)
    else:
        print(f"⚠️ File not found: {VAL_SPLIT_FILE}")

    # 3. Process Knowledge Base Split (sourced from train2017)
    if os.path.exists(KB_SPLIT_FILE):
        kb_data = read_json(KB_SPLIT_FILE)
        analyze_split_complexity("Knowledge Base Subset (kb_10k.json)", kb_data, train_counts_map)
    else:
        print(f"⚠️ File not found: {KB_SPLIT_FILE}")


if __name__ == "__main__":
    run_scene_complexity_analysis()

Loading COCO instance annotations...

   Scene Complexity Analysis: Test Subset (test_100.json)
Total Images Analyzed: 100
--------------------------------------------------
📊 Categorical Breakdown:
  • Simple Scenes (1-2 objects): 52    (52.00%)
  • Complex Scenes (>=3 objects): 48    (48.00%)
--------------------------------------------------
📈 Statistical Metrics:
  • Mean (Average) Objects / Image: 5.41
  • Median Objects / Image:        2.00
  • Min Objects in an Image:       1
  • Max Objects in an Image:       56

   Scene Complexity Analysis: Test Subset (test_1000.json)
Total Images Analyzed: 1000
--------------------------------------------------
📊 Categorical Breakdown:
  • Simple Scenes (1-2 objects): 500   (50.00%)
  • Complex Scenes (>=3 objects): 500   (50.00%)
--------------------------------------------------
📈 Statistical Metrics:
  • Mean (Average) Objects / Image: 5.67
  • Median Objects / Image:        2.50
  • Min Objects in an Image:       1
  • Max Objects in an

## Textual statistics

In [ ]:
from collections import Counter
import json
import os
import statistics
import nltk

# Ensure NLTK resources are downloaded
# We are going to use NLTK to tokenize the captions
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

# Paths for Google Colab environment
TEST_SPLIT_FILE = os.path.join("splits", "test_100.json")
TEST_1000_SPLIT_FILE = os.path.join("splits", "test_1000.json")
VAL_SPLIT_FILE = os.path.join("splits", "val_100.json")
KB_SPLIT_FILE = os.path.join("splits", "kb_10k.json")


def read_json(file_path):
    """Utility function to load a JSON file."""
    with open(file_path, "r") as f:
        return json.load(f)


def map_pos_tag(tag):
    """Map Penn Treebank POS tags to simplified syntactic categories."""
    if tag.startswith("NN"):
        return "Nouns (NN)"
    elif tag.startswith("VB"):
        return "Verbs (VB)"
    elif tag.startswith("JJ"):
        return "Adjectives (JJ)"
    elif tag == "IN":
        return "Prepositions / Subordinating Conjunctions (IN)"
    elif tag.startswith("RB"):
        return "Adverbs (RB)"
    elif tag in ["DT", "PDT", "WDT"]:
        return "Determiners (DT)"
    elif tag in ["PRP", "PRP$", "WP", "WP$"]:
        return "Pronouns (PRP)"
    else:
        return "Others"


def analyze_textual_statistics(split_name, split_data):
    """Calculates caption length distribution, vocabulary size, and POS distribution."""
    all_captions = []

    # Extract all reference captions
    for item in split_data:
        all_captions.extend(item.get("captions", []))

    total_captions = len(all_captions)
    if total_captions == 0:
        print(f"⚠️ No captions found in {split_name}")
        return

    caption_lengths = []
    all_tokens = []
    pos_counts = Counter()

    for caption in all_captions:
        # Tokenization (lowercased)
        tokens = nltk.word_tokenize(caption.lower())
        # Filter tokens and keep only real words, exclude punctuations
        word_tokens = [t for t in tokens if t.isalnum()]

        caption_lengths.append(len(word_tokens))
        all_tokens.extend(word_tokens)

        # POS Tagging
        # Tag each word
        tagged_words = nltk.pos_tag(word_tokens)
        for _, tag in tagged_words:
            # counts occurrence per tag
            simplified_pos = map_pos_tag(tag)
            pos_counts[simplified_pos] += 1

    total_words = len(all_tokens)
    # remove duplications using set to calculate vocab size
    vocab_size = len(set(all_tokens))

    # Length metrics
    mean_len = statistics.mean(caption_lengths)
    median_len = statistics.median(caption_lengths)
    min_len = min(caption_lengths)
    max_len = max(caption_lengths)

    # Display results
    print(f"\n==================================================")
    print(f"   Textual Statistics Analysis: {split_name}")
    print(f"==================================================")
    print(f"Total Captions Analyzed: {total_captions}")
    print(f"Total Word Tokens:       {total_words}")
    print(f"Unique Vocabulary Size:  {vocab_size} unique words")
    print("-" * 50)
    print("📏 Caption Length Distribution (Words / Caption):")
    print(f"  • Mean (Average) Length: {mean_len:.2f} words")
    print(f"  • Median Length:         {median_len:.2f} words")
    print(f"  • Min Length:            {min_len} words")
    print(f"  • Max Length:            {max_len} words")
    print("-" * 50)
    print("🏷️ Part-of-Speech (POS) Distribution:")
    print(f"{'POS Category':<42} | {'Count':<10} | {'Percentage':<10}")
    print("-" * 68)
    for pos_cat, count in pos_counts.most_common():
        pct = (count / total_words) * 100 if total_words > 0 else 0
        print(f"{pos_cat:<42} | {count:<10} | {pct:.2f}%")


def run_textual_statistics_analysis():
    print("Initializing NLTK resources and loading dataset splits...")

    # 1. Process Test Split
    if os.path.exists(TEST_SPLIT_FILE):
        test_data = read_json(TEST_SPLIT_FILE)
        analyze_textual_statistics("Test Subset (test_100.json)", test_data)
    else:
        print(f"⚠️ File not found: {TEST_SPLIT_FILE}")

    # 1b. Process Test 1000 Split
    if os.path.exists(TEST_1000_SPLIT_FILE):
        test_1000_data = read_json(TEST_1000_SPLIT_FILE)
        analyze_textual_statistics("Test Subset (test_1000.json)", test_1000_data)
    else:
        print(f"⚠️ File not found: {TEST_1000_SPLIT_FILE}")

    # 2. Process Validation Split
    if os.path.exists(VAL_SPLIT_FILE):
        val_data = read_json(VAL_SPLIT_FILE)
        analyze_textual_statistics("Validation Subset (val_100.json)", val_data)
    else:
        print(f"⚠️ File not found: {VAL_SPLIT_FILE}")

    # 3. Process Knowledge Base Split
    if os.path.exists(KB_SPLIT_FILE):
        kb_data = read_json(KB_SPLIT_FILE)
        analyze_textual_statistics("Knowledge Base Subset (kb_10k.json)", kb_data)
    else:
        print(f"⚠️ File not found: {KB_SPLIT_FILE}")


if __name__ == "__main__":
    run_textual_statistics_analysis()

Initializing NLTK resources and loading dataset splits...

   Textual Statistics Analysis: Test Subset (test_100.json)
Total Captions Analyzed: 500
Total Word Tokens:       5125
Unique Vocabulary Size:  959 unique words
--------------------------------------------------
📏 Caption Length Distribution (Words / Caption):
  • Mean (Average) Length: 10.25 words
  • Median Length:         10.00 words
  • Min Length:            7 words
  • Max Length:            30 words
--------------------------------------------------
🏷️ Part-of-Speech (POS) Distribution:
POS Category                               | Count      | Percentage
--------------------------------------------------------------------
Nouns (NN)                                 | 1776       | 34.65%
Determiners (DT)                           | 1027       | 20.04%
Prepositions / Subordinating Conjunctions (IN) | 854        | 16.66%
Verbs (VB)                                 | 602        | 11.75%
Adjectives (JJ)                         

## Diversity analysis

In [6]:
from collections import Counter
import json
import nltk
from nltk.tokenize import word_tokenize


# ============================================================
# 1. CONFIGURATION
# ============================================================

TRAIN_INSTANCES_FILE = "annotations/instances_train2017.json"
VAL_INSTANCES_FILE = "annotations/instances_val2017.json"

TEST_100_FILE = "splits/test_100.json"
TEST_1000_FILE = "splits/test_1000.json"
VAL_100_FILE = "splits/val_100.json"
KB_10K_FILE = "splits/kb_10k.json"


# Download NLTK tokenizer resources
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)


# ============================================================
# 2. KEYWORD LEXICONS
# ============================================================

COLOR_KEYWORDS = {
    "white", "black", "red", "blue", "green",
    "yellow", "brown", "orange", "grey", "gray",
    "pink", "purple", "tan", "blond", "silver", "gold"
}

SHAPE_KEYWORDS = {
    "round", "square", "curved", "flat", "oval",
    "circular", "triangular", "rectangular", "straight"
}

DIMENSION_KEYWORDS = {
    "large", "small", "tall", "short", "long",
    "wide", "narrow", "big", "little", "tiny",
    "huge", "deep", "thick", "thin"
}

DAY_KEYWORDS = {
    "day", "daylight", "sunny", "sunlight",
    "sunshine", "morning", "afternoon", "sunrise"
}

NIGHT_KEYWORDS = {
    "night", "nighttime", "midnight",
    "moon", "moonlight", "moonlit"
}

TWILIGHT_KEYWORDS = {
    "dusk", "twilight", "sunset", "evening"
}


# Environment keywords
INDOOR_KEYWORDS = {
    "room", "kitchen", "bedroom", "inside"
}

OUTDOOR_KEYWORDS = {
    "street", "sky", "park", "field",
    "outside", "beach"
}

INDOOR_PHRASES = {
    ("living", "room")
}


# COCO supercategories
INDOOR_SUPERCATEGORIES = {
    "furniture",
    "kitchen",
    "indoor",
    "appliance",
    "electronic"
}

OUTDOOR_SUPERCATEGORIES = {
    "outdoor",
    "vehicle",
    "animal",
    "sports"
}


# ============================================================
# 3. JSON LOADING
# ============================================================

def read_json(file_path):
    """Load and return a JSON file."""

    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)


# ============================================================
# 4. CATEGORY CONTEXT MAPPING
# ============================================================

def map_category_contexts(instances_data):
    """
    Map each COCO category ID to an environment context:
    indoor, outdoor, or other.
    """

    category_contexts = {}

    for category in instances_data["categories"]:

        category_id = category["id"]
        supercategory = category["supercategory"].lower()

        if supercategory in INDOOR_SUPERCATEGORIES:
            category_contexts[category_id] = "indoor"

        elif supercategory in OUTDOOR_SUPERCATEGORIES:
            category_contexts[category_id] = "outdoor"

        else:
            category_contexts[category_id] = "other"

    return category_contexts


# ============================================================
# 5. IMAGE-CATEGORY MAPPING
# ============================================================

def build_image_categories_map(instances_data):
    """
    Build a mapping:
        image_id -> list of category IDs

    Each category ID represents an object instance annotation.
    A category may appear multiple times for an image.
    """

    image_categories = {}

    for annotation in instances_data["annotations"]:

        image_id = annotation["image_id"]
        category_id = annotation["category_id"]

        image_categories.setdefault(image_id, []).append(category_id)

    return image_categories


# ============================================================
# 6. TEXT TOKENIZATION
# ============================================================

def tokenize_captions(captions):
    """
    Combine captions, lowercase the text, and tokenize it
    using NLTK.

    Punctuation-only tokens are removed.
    """

    combined_text = " ".join(captions).lower()

    tokens = word_tokenize(combined_text)

    words = [
        token
        for token in tokens
        if any(character.isalnum() for character in token)
    ]

    return words, combined_text


# ============================================================
# 7. PHRASE MATCHING
# ============================================================

def contains_phrase(words, phrase):
    """
    Check whether a sequence of words contains an exact
    consecutive phrase.
    """

    phrase_length = len(phrase)

    for index in range(len(words) - phrase_length + 1):

        if tuple(words[index:index + phrase_length]) == phrase:
            return True

    return False


# ============================================================
# 8. ENVIRONMENT CLASSIFICATION
# ============================================================

def classify_environment(
    image_id,
    image_categories,
    category_contexts,
    words
):
    """
    Classify an image as indoor, outdoor, or unclear.

    Priority:
        1. COCO object-category context.
        2. Caption keyword evidence if category votes are tied.

    Exact token matching avoids substring errors.
    """

    category_ids = image_categories.get(image_id, [])

    context_counts = Counter()

    for category_id in category_ids:

        context = category_contexts.get(category_id, "other")

        if context in {"indoor", "outdoor"}:
            context_counts[context] += 1

    indoor_count = context_counts["indoor"]
    outdoor_count = context_counts["outdoor"]

    if indoor_count > outdoor_count:
        return "indoor"

    if outdoor_count > indoor_count:
        return "outdoor"

    # Category evidence is tied or unavailable.
    # Use exact whole-token matching as a fallback.

    word_set = set(words)

    has_indoor = bool(word_set.intersection(INDOOR_KEYWORDS))
    has_outdoor = bool(word_set.intersection(OUTDOOR_KEYWORDS))

    if any(
        contains_phrase(words, phrase)
        for phrase in INDOOR_PHRASES
    ):
        has_indoor = True

    if has_indoor and not has_outdoor:
        return "indoor"

    if has_outdoor and not has_indoor:
        return "outdoor"

    return "unclear"


# ============================================================
# 9. DAY / NIGHT CLASSIFICATION
# ============================================================

def classify_lighting(words):
    """
    Classify lighting/time-of-day cues from caption tokens.

    Categories:
        day
        night
        mixed
        unspecified
    """

    word_set = set(words)

    has_day = bool(word_set.intersection(DAY_KEYWORDS))
    has_night = bool(word_set.intersection(NIGHT_KEYWORDS))
    has_twilight = bool(word_set.intersection(TWILIGHT_KEYWORDS))

    if has_day and has_night:
        return "mixed"

    if has_twilight:
        return "mixed"

    if has_day:
        return "day"

    if has_night:
        return "night"

    return "unspecified"


# ============================================================
# 10. STATISTICS HELPER FUNCTIONS
# ============================================================

def calculate_percentage(count, total):
    """Calculate percentage safely."""

    if total == 0:
        return 0.0

    return (count / total) * 100


def print_distribution(title, counter, categories, total):
    """
    Print the complete distribution of predefined categories.

    Categories with zero occurrences are also displayed.
    """

    print(f"\n--- {title} ---")

    for category in categories:

        count = counter[category]
        percentage = calculate_percentage(count, total)

        print(
            f"{category.capitalize():20s} "
            f"{count:8d} "
            f"({percentage:6.2f}%)"
        )


def print_attribute_statistics(
    title,
    counter,
    keywords,
    image_presence_counter,
    total_images
):
    """
    Print statistics for every keyword in an attribute lexicon.

    For each keyword:
        - Total number of mentions.
        - Number of images containing the keyword.
        - Percentage of images containing the keyword.

    All keywords are displayed, including zero-occurrence keywords.
    """

    print("\n" + "-" * 75)
    print(title)
    print("-" * 75)

    print(
        f"{'Keyword':20s}"
        f"{'Mentions':15s}"
        f"{'Images':15s}"
        f"{'Image %':12s}"
    )

    print("-" * 75)

    total_mentions = 0
    total_images_with_attribute = 0

    for keyword in sorted(keywords):

        mentions = counter[keyword]
        image_count = image_presence_counter[keyword]

        image_percentage = calculate_percentage(
            image_count,
            total_images
        )

        total_mentions += mentions

        if mentions > 0:
            total_images_with_attribute += image_count

        print(
            f"{keyword:20s}"
            f"{mentions:<15d}"
            f"{image_count:<15d}"
            f"{image_percentage:>7.2f}%"
        )

    print("-" * 75)

    print(f"Total mentions: {total_mentions}")

    print(
        f"Images containing at least one {title.lower()} keyword: "
        f"{total_images_with_attribute} / {total_images}"
    )

    print()


# ============================================================
# 11. SPLIT DIVERSITY ANALYSIS
# ============================================================

def analyze_split_diversity(
    split_name,
    split_file,
    image_categories,
    category_contexts
):
    """
    Analyze the diversity of a dataset split.

    Statistics:
        - Total images
        - Total captions
        - Color mentions and image presence
        - Shape mentions and image presence
        - Dimension mentions and image presence
        - Environment distribution
        - Day/night distribution
    """

    print("\n" + "=" * 85)
    print(f"DIVERSITY ANALYSIS: {split_name}")
    print("=" * 85)

    split_data = read_json(split_file)

    total_images = len(split_data)

    total_captions = sum(
        len(item["captions"])
        for item in split_data
    )

    # --------------------------------------------------------
    # Initialize attribute mention counters
    # --------------------------------------------------------

    color_counter = Counter()
    shape_counter = Counter()
    dimension_counter = Counter()

    # --------------------------------------------------------
    # Initialize image-presence counters
    # --------------------------------------------------------

    color_image_counter = Counter()
    shape_image_counter = Counter()
    dimension_image_counter = Counter()

    # --------------------------------------------------------
    # Initialize classification counters
    # --------------------------------------------------------

    environment_counter = Counter()
    lighting_counter = Counter()

    # ========================================================
    # LOOP 1: PROCESS EACH IMAGE IN THE SPLIT
    # ========================================================

    for item in split_data:

        image_id = item["image_id"]
        captions = item["captions"]

        # ----------------------------------------------------
        # Tokenize captions
        # ----------------------------------------------------

        words, combined_text = tokenize_captions(captions)

        word_counter = Counter(words)

        # ----------------------------------------------------
        # Count color mentions and image presence
        # ----------------------------------------------------

        for keyword in COLOR_KEYWORDS:

            mentions = word_counter[keyword]

            if mentions > 0:

                color_counter[keyword] += mentions
                color_image_counter[keyword] += 1

        # ----------------------------------------------------
        # Count shape mentions and image presence
        # ----------------------------------------------------

        for keyword in SHAPE_KEYWORDS:

            mentions = word_counter[keyword]

            if mentions > 0:

                shape_counter[keyword] += mentions
                shape_image_counter[keyword] += 1

        # ----------------------------------------------------
        # Count dimension mentions and image presence
        # ----------------------------------------------------

        for keyword in DIMENSION_KEYWORDS:

            mentions = word_counter[keyword]

            if mentions > 0:

                dimension_counter[keyword] += mentions
                dimension_image_counter[keyword] += 1

        # ----------------------------------------------------
        # Classify environment
        # ----------------------------------------------------

        environment = classify_environment(
            image_id,
            image_categories,
            category_contexts,
            words
        )

        environment_counter[environment] += 1

        # ----------------------------------------------------
        # Classify lighting
        # ----------------------------------------------------

        lighting = classify_lighting(words)

        lighting_counter[lighting] += 1

    # ========================================================
    # 12. PRINT COMPLETE STATISTICS
    # ========================================================

    print("\n" + "=" * 85)
    print("GENERAL STATISTICS")
    print("=" * 85)

    print(f"Split name:       {split_name}")
    print(f"Split file:       {split_file}")
    print(f"Total images:     {total_images}")
    print(f"Total captions:   {total_captions}")

    if total_images > 0:

        print(
            f"Average captions per image: "
            f"{total_captions / total_images:.2f}"
        )

    # --------------------------------------------------------
    # Color statistics
    # --------------------------------------------------------

    print_attribute_statistics(
        "COLOR STATISTICS",
        color_counter,
        COLOR_KEYWORDS,
        color_image_counter,
        total_images
    )

    # --------------------------------------------------------
    # Shape statistics
    # --------------------------------------------------------

    print_attribute_statistics(
        "SHAPE STATISTICS",
        shape_counter,
        SHAPE_KEYWORDS,
        shape_image_counter,
        total_images
    )

    # --------------------------------------------------------
    # Dimension statistics
    # --------------------------------------------------------

    print_attribute_statistics(
        "DIMENSION STATISTICS",
        dimension_counter,
        DIMENSION_KEYWORDS,
        dimension_image_counter,
        total_images
    )

    # --------------------------------------------------------
    # Environment distribution
    # --------------------------------------------------------

    print_distribution(
        "ENVIRONMENT DISTRIBUTION",
        environment_counter,
        ["indoor", "outdoor", "unclear"],
        total_images
    )

    # --------------------------------------------------------
    # Lighting distribution
    # --------------------------------------------------------

    print_distribution(
        "LIGHTING / TIME-OF-DAY DISTRIBUTION",
        lighting_counter,
        ["day", "night", "mixed", "unspecified"],
        total_images
    )


# ============================================================
# 13. MAIN ANALYSIS PIPELINE
# ============================================================

def run_diversity_analysis():

    # --------------------------------------------------------
    # Load instance annotations
    # --------------------------------------------------------

    print("Loading COCO annotations...")

    train_instances = read_json(TRAIN_INSTANCES_FILE)
    val_instances = read_json(VAL_INSTANCES_FILE)

    # --------------------------------------------------------
    # Build category context mappings
    # --------------------------------------------------------

    train_category_contexts = map_category_contexts(
        train_instances
    )

    val_category_contexts = map_category_contexts(
        val_instances
    )

    # --------------------------------------------------------
    # Build image-category mappings
    # --------------------------------------------------------

    train_image_categories = build_image_categories_map(
        train_instances
    )

    val_image_categories = build_image_categories_map(
        val_instances
    )

    # ========================================================
    # ANALYZE TEST 100
    # ========================================================

    analyze_split_diversity(
        "TEST 100",
        TEST_100_FILE,
        val_image_categories,
        val_category_contexts
    )

    # ========================================================
    # ANALYZE TEST 1000
    # ========================================================

    analyze_split_diversity(
        "TEST 1000",
        TEST_1000_FILE,
        val_image_categories,
        val_category_contexts
    )

    # ========================================================
    # ANALYZE VALIDATION 100
    # ========================================================

    analyze_split_diversity(
        "VALIDATION 100",
        VAL_100_FILE,
        val_image_categories,
        val_category_contexts
    )

    # ========================================================
    # ANALYZE KNOWLEDGE BASE 10K
    # ========================================================

    analyze_split_diversity(
        "KNOWLEDGE BASE 10K",
        KB_10K_FILE,
        train_image_categories,
        train_category_contexts
    )


# ============================================================
# 14. SCRIPT ENTRY POINT
# ============================================================

if __name__ == "__main__":
    run_diversity_analysis()

Loading COCO annotations...

DIVERSITY ANALYSIS: TEST 100

GENERAL STATISTICS
Split name:       TEST 100
Split file:       splits/test_100.json
Total images:     100
Total captions:   500
Average captions per image: 5.00

---------------------------------------------------------------------------
COLOR STATISTICS
---------------------------------------------------------------------------
Keyword             Mentions       Images         Image %     
---------------------------------------------------------------------------
black               13             9                 9.00%
blond               0              0                 0.00%
blue                13             7                 7.00%
brown               9              4                 4.00%
gold                1              1                 1.00%
gray                1              1                 1.00%
green               10             5                 5.00%
grey                0              0                 0.00